# 001 · The sys Module

> **Standard Library › System & CLI** · Python 3.14

`sys` is the interpreter's control panel: command-line arguments, exit codes, the standard streams, the module search path and facts about the running Python. In .NET this information is spread across `Environment`, `Console`, `RuntimeInformation` and `AppDomain`.

**In this notebook you will learn:**

- `sys.argv` and how it differs from C#'s `args`
- `sys.exit()`, `SystemExit` and exit codes
- the standard streams `sys.stdin`, `sys.stdout` and `sys.stderr`
- `sys.path` and `sys.modules` — where imports come from and where they are cached
- version and platform facts, `sys.getsizeof` and the recursion limit

## .NET mental model

| .NET | Python `sys` |
|---|---|
| `string[] args` / `Environment.GetCommandLineArgs()` | `sys.argv` |
| `Environment.Exit(code)` / `return code;` from `Main` | `sys.exit(code)` (raises `SystemExit`) |
| `Console.In` / `Console.Out` / `Console.Error` | `sys.stdin` / `sys.stdout` / `sys.stderr` |
| assembly probing paths | `sys.path` |
| `AppDomain.CurrentDomain.GetAssemblies()` | `sys.modules` |
| `Environment.Version` / `RuntimeInformation.FrameworkDescription` | `sys.version` / `sys.version_info` |
| `Environment.ProcessPath` | `sys.executable` |
| `OperatingSystem.IsWindows()` | `sys.platform == "win32"` |
| (no equivalent — stack overflow kills the process) | `sys.getrecursionlimit()` |

`sys` is always available (it is built into the interpreter), but you still have to `import sys`.

## `sys.argv` — command-line arguments

`sys.argv` is a plain `list[str]`. Unlike C#'s `args`, **index 0 is the script name**, so the real arguments start at `sys.argv[1]` — the same layout as `Environment.GetCommandLineArgs()`.

```csharp
// C#: dotnet run -- alice 42
static void Main(string[] args)
{
    Console.WriteLine(args[0]);                              // alice
    Console.WriteLine(Environment.GetCommandLineArgs()[0]);  // path of the program
}
```

Inside Jupyter, `sys.argv` belongs to the *kernel* process, so it is not very interesting (we only print the file name, not the full path):

In [ ]:
import sys
from pathlib import Path

print(type(sys.argv).__name__, len(sys.argv))
print(Path(sys.argv[0]).name)        # the kernel's launcher script

To see `sys.argv` in a real script, we write a tiny script into a temporary directory and run it as a separate process with `subprocess` *(see Standard Library › System & CLI › 004 · Running Processes with subprocess)*. `sys.executable` is the path of the Python running this notebook, so the child uses the same interpreter.

In [ ]:
import subprocess
import tempfile

script_source = """
import sys
from pathlib import Path
print("script:", Path(sys.argv[0]).name)
print("args:  ", sys.argv[1:])
"""

with tempfile.TemporaryDirectory() as tmp:
    script = Path(tmp) / "greet.py"
    script.write_text(script_source)
    result = subprocess.run([sys.executable, str(script), "alice", "42", "--loud"],
                            capture_output=True, text=True)
    print(result.stdout)

Every argument is a **string** — convert and validate them yourself, or (much better) use `argparse` *(see Standard Library › System & CLI › 003 · Command-Line Arguments with argparse)*.

## `sys.exit()` and `SystemExit`

`sys.exit(code)` does not kill the process immediately — it **raises `SystemExit`**. The exception unwinds the stack (running `finally` blocks and `with` exits) and, if nothing catches it, the interpreter exits with that code.

| Call | Exit code | Notes |
|---|---|---|
| `sys.exit()` / `sys.exit(0)` / `sys.exit(None)` | 0 | success |
| `sys.exit(3)` | 3 | any int |
| `sys.exit("fatal: config missing")` | 1 | the message is printed to stderr |

```csharp
// C#
Environment.Exit(3);   // terminates now; finally blocks in the calling code do NOT run
```

In [ ]:
try:
    sys.exit(3)
except SystemExit as e:
    print(f"{type(e).__name__}: code={e.code}")

try:
    sys.exit("fatal: config missing")
except SystemExit as e:
    print(f"{type(e).__name__}: code={e.code!r}")   # the message is the "code"

Because it is an exception, cleanup code runs:

In [ ]:
def main() -> None:
    try:
        print("working…")
        sys.exit(2)
    finally:
        print("finally still runs")   # unlike Environment.Exit

try:
    main()
except SystemExit as e:
    print("exit code:", e.code)

⚠️ **Gotcha:** `SystemExit` derives from `BaseException`, **not** `Exception`, so `except Exception:` does not swallow it (good!). A bare `except:` or `except BaseException:` does — avoid those.

In [ ]:
print(issubclass(SystemExit, Exception))       # False
print(issubclass(SystemExit, BaseException))   # True

try:
    try:
        sys.exit(1)
    except Exception:
        print("never printed")
except SystemExit:
    print("SystemExit passed straight through 'except Exception'")

Here is what the operating system sees, using child processes. (The Jupyter kernel sets `FORCE_COLOR=1`, which child Pythons inherit and use to colour their tracebacks; setting `PYTHON_COLORS=0` turns that off so the captured text stays plain.)

In [ ]:
import os

plain_env = {**os.environ, "PYTHON_COLORS": "0"}
for code in ["sys.exit()", "sys.exit(3)", "sys.exit('fatal: config missing')", "1 / 0"]:
    child = subprocess.run([sys.executable, "-c", f"import sys; {code}"],
                           capture_output=True, text=True, env=plain_env)
    last_err_line = child.stderr.strip().splitlines()[-1] if child.stderr.strip() else ""
    print(f"{code:36} → returncode={child.returncode}  stderr={last_err_line!r}")

An uncaught exception exits with code **1** and prints the traceback to stderr. A common, clean script layout is:

```python
def main(argv: list[str] | None = None) -> int:
    ...
    return 0

if __name__ == "__main__":
    sys.exit(main())          # like: static int Main(string[] args)
```

**Tip:** `os._exit(code)` is the hard exit: it stops the process immediately, without running `finally` blocks, `atexit` handlers or flushing buffers — closer to .NET's "stop now" `Environment.Exit` behaviour, only harsher. You almost never want it outside forked child processes.

## Standard streams: `stdin`, `stdout`, `stderr`

`sys.stdout` and `sys.stderr` are text file objects, like `Console.Out` and `Console.Error`. `print()` writes to `sys.stdout` by default; pass `file=sys.stderr` for errors and diagnostics.

```csharp
// C#
Console.Out.WriteLine("result");
Console.Error.WriteLine("warning: something odd");
```

In [ ]:
print("result")                                  # → stdout
print("warning: something odd", file=sys.stderr) # → stderr (Jupyter shows it highlighted)

count = sys.stdout.write("written without a newline\n")   # returns characters written
print(count)

Keeping data on stdout and messages on stderr lets users pipe your program's output (`python tool.py > out.txt`) while still seeing errors.

`sys.stdin` is read like a file. In a script you would typically write one of these (not run here, as a notebook has no interactive stdin):

```python
for line in sys.stdin:            # streams line by line, like Console.In.ReadLine() in a loop
    process(line.rstrip("\n"))

data = sys.stdin.read()           # everything at once, like Console.In.ReadToEnd()
```

We can still demonstrate it by feeding a child process's stdin:

In [ ]:
upper = "import sys\nfor line in sys.stdin:\n    print(line.strip().upper())"
child = subprocess.run([sys.executable, "-c", upper],
                       input="alice\nbob\n", capture_output=True, text=True)
print(child.stdout)

## `sys.path` — where imports are searched

`sys.path` is a list of directories (and zip files) that `import` searches, in order: the script's directory, `PYTHONPATH` entries, the standard library, then `site-packages` (your installed packages). It plays the role of .NET's assembly probing paths, but it is a **mutable list** at runtime.

In [ ]:
print(type(sys.path).__name__, len(sys.path) > 0)
print(any(Path(p).name == "site-packages" for p in sys.path))   # True — installed packages
# (we don't print the entries: they are absolute, machine-specific paths)

⚠️ **Gotcha:** `sys.path.append(...)` hacks work, but they hide your project's structure. Prefer a proper package layout plus `uv` *(see Syntax › Modules & Packages › 003 · Environments and Dependencies with uv)*. And never name a file after a stdlib module (`random.py`, `json.py`) — the script directory comes first on `sys.path`, so it shadows the real module.

## `sys.modules` — the import cache

Every module is executed **once** per process; afterwards `import` just returns the cached object from the `sys.modules` dictionary. It is roughly "the assemblies already loaded into this AppDomain".

In [ ]:
import json

print("json" in sys.modules)                  # True
print(sys.modules["json"] is json)            # True — the very same module object
print(len(sys.modules) > 50)                  # lots of modules are loaded already

print("tomllib" in sys.stdlib_module_names)   # True — is a name part of the stdlib?

## Version and platform information

`sys.version_info` is a named tuple, so compare it with a tuple — never compare version *strings*.

```csharp
// C#
Console.WriteLine(Environment.Version);                       // runtime version
Console.WriteLine(RuntimeInformation.FrameworkDescription);   // ".NET 9.0.x"
if (OperatingSystem.IsWindows()) { /* ... */ }
```

In [ ]:
print(sys.version_info[:3])                    # e.g. (3, 14, 0)
print(sys.version_info >= (3, 14))             # True
print(sys.version.split()[0])                  # the version part of the long version string
print(sys.implementation.name)                 # cpython
print(sys.platform)                            # 'darwin', 'linux' or 'win32'
print(sys.byteorder, sys.maxsize == 2**63 - 1) # little True on 64-bit machines

⚠️ **Gotcha:** `sys.platform` is `"win32"` even on 64-bit Windows. Checking `sys.platform.startswith("linux")` is the recommended way to detect Linux. For richer details see *(see Standard Library › System & CLI › 002 · Environment Variables and Platform Info)*.

## `sys.executable`

The absolute path of the running interpreter — the reliable way to start another Python process with the *same* environment (virtual environment included). Avoid hard-coding `"python"`, which may resolve to a different installation.

In [ ]:
print(Path(sys.executable).name)              # e.g. python or python3 (full path not shown)
print(Path(sys.executable).is_file())         # True

## `sys.getsizeof` — shallow object size

`sys.getsizeof(obj)` returns the size in bytes of the object **itself** — not of the objects it references. Even small Python objects carry a header (reference count + type pointer), so they are bigger than their C# value-type counterparts (an `int` is 4 bytes in C#).

In [ ]:
print(sys.getsizeof(1))                 # 28 on 64-bit CPython — ints are full objects
print(sys.getsizeof(2**100))            # bigger: ints grow as needed (like BigInteger)
print(sys.getsizeof("abc"))             # string header + data
print(sys.getsizeof([]), sys.getsizeof([1, 2, 3]))   # list header + one pointer per item

big_strings = ["x" * 10_000] * 3
print(sys.getsizeof(big_strings))       # small! the strings themselves are not counted

**Tip:** for a "deep" size you must walk the object graph yourself; for real memory investigations use the `tracemalloc` module.

## The recursion limit

CPython limits Python-level recursion depth (default **1000**) and raises `RecursionError` instead of crashing. .NET has no such guard: a runaway recursion throws a `StackOverflowException` that cannot be caught and terminates the process.

In [ ]:
print(sys.getrecursionlimit())          # 1000

def depth(n: int) -> int:
    return 0 if n == 0 else 1 + depth(n - 1)

print(depth(500))                       # fine
try:
    depth(5_000)
except RecursionError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** `sys.setrecursionlimit(...)` can raise the limit, but deep recursion is still slow and memory-hungry in Python (there is no tail-call optimisation). Prefer rewriting deep recursion as a loop with an explicit stack.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Command-line args | `args` (no program name) | `sys.argv[1:]` |
| Program path in args | `Environment.GetCommandLineArgs()[0]` | `sys.argv[0]` |
| Exit with a code | `Environment.Exit(3)` / `return 3` from `Main` | `sys.exit(3)` |
| Exit with an error message | `Console.Error.WriteLine(...)` + `Exit(1)` | `sys.exit("message")` |
| Write to stderr | `Console.Error.WriteLine(...)` | `print(..., file=sys.stderr)` |
| Read all stdin | `Console.In.ReadToEnd()` | `sys.stdin.read()` |
| Module search path | probing paths | `sys.path` |
| Loaded modules | `AppDomain.GetAssemblies()` | `sys.modules` |
| Runtime version | `Environment.Version` | `sys.version_info` |
| Is Windows? | `OperatingSystem.IsWindows()` | `sys.platform == "win32"` |
| Interpreter path | `Environment.ProcessPath` | `sys.executable` |
| Object size | no direct equivalent | `sys.getsizeof(obj)` (shallow) |
| Recursion guard | none (stack overflow crashes) | `sys.getrecursionlimit()` → `RecursionError` |

## Exercises

1. Write `main(argv: list[str]) -> int` that sums its integer arguments and prints the total, returning `0`; if any argument is not an integer, print an error to **stderr** and return `2`. Call it with `["1", "2", "3"]` and `["1", "x"]`, then wrap one call in `try: sys.exit(main(...)) except SystemExit as e:` to show the exit code.
2. Use `subprocess.run([sys.executable, "-c", ...])` to run a child that prints `sys.argv[1:]`, passing the arguments `["hello world", "--flag"]`. How many arguments does the child see?
3. Check whether the `decimal` module is already loaded in this process, import it, and check again. Then confirm that importing it a second time returns the same object.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import subprocess
import sys

# 1 — return the exit code from main, let sys.exit turn it into SystemExit
def main(argv: list[str]) -> int:
    try:
        numbers = [int(a) for a in argv]
    except ValueError as e:
        print(f"error: {e}", file=sys.stderr)
        return 2
    print(sum(numbers))
    return 0

print(main(["1", "2", "3"]))        # prints 6, then 0
try:
    sys.exit(main(["1", "x"]))
except SystemExit as e:
    print("exit code:", e.code)     # 2

# 2 — list arguments are passed as-is: "hello world" stays ONE argument
child = subprocess.run(
    [sys.executable, "-c", "import sys; print(sys.argv[1:])", "hello world", "--flag"],
    capture_output=True, text=True,
)
print(child.stdout.strip())         # ['hello world', '--flag'] — 2 arguments

# 3 — (whether decimal is already loaded depends on what ran before)
print("decimal" in sys.modules)
import decimal
print("decimal" in sys.modules)     # True
import decimal as again
print(again is decimal is sys.modules["decimal"])   # True
```

</details>